In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "movie_cast"
v_esquema = "movie_silver"
v_tabla = "movies_casts"
v_partition = "file_date"
v_merge_condition = "target.movie_id = source.movie_id and target.person_id = source.person_id"

In [0]:
#1. leemos el archivo JSON multilinea
# Define el la estructura personName
movie_cast_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True)
])

# Cargamos el archivo utilizando la estructura definida
movie_cast_df = spark.read\
    .schema(movie_cast_schema)\
    .option("multiLine", "true")\
    .json(f"{bronze_folder_path}/{v_file_date}/{v_archivo}.json")

# Mostramos el resultado
display(movie_cast_df)


In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
movie_cast_renamed_df = movie_cast_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("personId", "person_id")\
    .withColumnRenamed("characterName", "character_name")

movie_cast_final_df = movie_cast_renamed_df.select(col("movie_id"), col("person_id"), col("character_name"))


movie_cast_final_df = add_ingestion_date(movie_cast_final_df)
movie_cast_final_df = add_env(movie_cast_final_df)
final_df = add_file_date (movie_cast_final_df)

display(final_df)

In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
resultado = merge_delta_lake (v_esquema, v_tabla, final_df, v_merge_condition, v_partition)
print(resultado)

In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 

#movie_cast_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_casts")
#movie_cast_final_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
#print(f"Se insertaron {movie_cast_final_df.count()} registros en la tabla {v_esquema}.{v_tabla}")


In [0]:
dbutils.notebook.exit("El notebook 07. Ingestion File movie_cast, termino correctamente")